# nuScenes vehicle trajectory prediction: Flow Matching baseline → Y-Flow

VESSL JupyterLab에서 위에서부터 실행. `PROFILE='mini'`로 연결 확인 후 `'trainval'`로 본 실험.
`SMOKE=True`면 데이터 없이 synthetic으로 전체 파이프라인만 점검.

In [ ]:
PROFILE = 'trainval'
SMOKE = False
DEVICE = 'cuda'
TRIALS = 40
STEPS = None
NUSCENES_META_URL = ''
NUSCENES_MAP_URL = ''

In [ ]:
import os, sys, json, subprocess
from pathlib import Path
ROOT = Path.cwd()
assert (ROOT / 'run_all.py').exists(), 'run this notebook from the nusc_yflow folder'
if NUSCENES_META_URL: os.environ['NUSCENES_META_URL'] = NUSCENES_META_URL
if NUSCENES_MAP_URL: os.environ['NUSCENES_MAP_URL'] = NUSCENES_MAP_URL
def sh(*a): print('+', ' '.join(a), flush=True); subprocess.run([sys.executable, '-m', *a], cwd=ROOT, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], cwd=ROOT, check=True)
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

## 1. 테스트

In [ ]:
sh('pytest', 'tests', '-q')

## 2. 데이터 다운로드 + 변환 (metadata + map expansion만, 센서 blob 불필요)

In [ ]:
if SMOKE:
    CACHE, CFG, SPLITS = 'datasets/synthetic', 'configs/synthetic.yaml', ('train', 'val', 'val')
    if not (ROOT / CACHE / 'train.npz').exists(): sh('tools.make_synthetic', '--out', CACHE, '--n_train', '4000', '--n_val', '500')
else:
    VERSION = f'v1.0-{PROFILE}'
    if not (ROOT / 'data/nuscenes' / VERSION).exists(): sh('tools.download_nuscenes', '--root', 'data/nuscenes', '--profile', PROFILE)
    CACHE, CFG = f'datasets/nuscenes_{PROFILE}', 'configs/nuscenes.yaml'
    SPLITS = ('mini_train', 'mini_val', 'mini_val') if PROFILE == 'mini' else ('train', 'train_val', 'val')
    if not all((ROOT / CACHE / f'{s}.npz').exists() for s in SPLITS):
        sh('tools.convert_nuscenes', '--root', 'data/nuscenes', '--version', VERSION, '--out', CACHE, '--splits', *dict.fromkeys(SPLITS))
print(json.load(open(ROOT / CACHE / 'meta.json')))

## 3. 공통 설정 + GT 제약 audit

`viol_rate_*`가 ~0이 되도록 `configs/nuscenes.yaml`의 `constraints.*` threshold를 조정한 뒤 진행.

In [ ]:
import yaml
RUN = 'fm_synthetic' if SMOKE else f'fm_{PROFILE}'
COMMON = [f'device={DEVICE}', f'data.cache_dir={CACHE}', f'data.train_split={SPLITS[0]}', f'data.eval_split={SPLITS[1]}', f'run_name={RUN}']
if STEPS: COMMON += [f'train.steps={STEPS}', f'train.save_every={max(100, STEPS // 10)}']
if PROFILE == 'mini' and not SMOKE: COMMON += ['train.batch_size=32', 'train.select_limit=500', 'data.limit_tune=300']
if SMOKE:
    base = yaml.safe_load(open(ROOT / 'configs/nuscenes.yaml')); syn = yaml.safe_load(open(ROOT / CFG))
    syn['constraints'] = {**base['constraints'], 'hard': ['speed', 'continuity', 'accel']}; syn['yflow'] = base['yflow']
    CFG = 'configs/smoke_constraints.yaml'; yaml.safe_dump(syn, open(ROOT / CFG, 'w'))
    COMMON += ['train.steps=600', 'train.save_every=300', 'train.batch_size=64', 'model.d_model=64', 'model.n_heads=4',
               'model.enc_layers=1', 'model.dec_layers=1', 'train.select_limit=200', 'data.limit_tune=100', 'eval.batch_size=100']
RUN_DIR = ROOT / 'outputs' / RUN
sh('vfm.evaluate', '--config', CFG, '--method', 'gt', *COMMON)
sh('tools.audit_gt', '--cache', CACHE, '--split', SPLITS[0])

## 4. Flow Matching baseline 학습 (best.pt = train_val minADE 최소 EMA 체크포인트)

In [ ]:
if not (RUN_DIR / 'last.pt').exists(): sh('vfm.train', '--config', CFG, *COMMON)
for r in [json.loads(l) for l in open(RUN_DIR / 'train_log.jsonl')][-5:]: print(r)

## 5. Baseline 평가 (CV vs FM, model-selection split)

In [ ]:
sh('vfm.evaluate', '--config', CFG, '--method', 'cv', *COMMON)
sh('vfm.evaluate', '--config', CFG, '--method', 'fm', *COMMON)

## 6. Y-Flow 하이퍼파라미터 탐색 (train_val 일부, objective = minADE + 5·(1−all_hard_safe) + MR)

In [ ]:
if not (RUN_DIR / 'tune/best.yaml').exists(): sh('vfm.tune', '--config', CFG, '--trials', str(TRIALS), *COMMON)
rows = [json.loads(l) for l in open(RUN_DIR / 'tune/trials.jsonl') if l.strip()]
ok = sorted([r for r in rows if 'objective' in r], key=lambda r: r['objective'])
for r in ok[:5]: print(round(r['objective'], 3), {k: (round(v, 3) if isinstance(v, float) else v) for k, v in r['params'].items()})
print(yaml.safe_load(open(RUN_DIR / 'tune/best.yaml'))['yflow'])

## 7. 최종 평가 (held-out split: val) — CV / FM / Y-Flow(best)

In [ ]:
FINAL = [f'data.eval_split={SPLITS[2]}', 'eval.save_predictions=true', 'data.limit_eval=null']
sh('vfm.evaluate', '--config', CFG, '--method', 'cv', *COMMON, *FINAL)
sh('vfm.evaluate', '--config', CFG, '--method', 'fm', *COMMON, *FINAL)
sh('vfm.evaluate', '--config', str(RUN_DIR / 'tune/best.yaml'), '--method', 'yflow', '--ref_pred', str(RUN_DIR / f'pred_fm_{SPLITS[2]}_seed0.npy'), *FINAL)
import pandas as pd
rows = {m: json.load(open(RUN_DIR / f'metrics_{m}_{SPLITS[2]}_seed0.json')) for m in ('cv', 'fm', 'yflow')}
keys = [k for k in rows['fm'] if k.startswith(('minADE', 'minFDE', 'MR_', 'all_hard_safe', 'viol_rate', 'mode_lost', 'ms_per'))]
df = pd.DataFrame({m: {k: r.get(k) for k in keys} for m, r in rows.items()}).T
(RUN_DIR / 'final_summary.json').write_text(json.dumps(rows, indent=2)); df

## 8. 시각화 (샘플 몇 개)

In [ ]:
import numpy as np, matplotlib
matplotlib.use('Agg'); import matplotlib.pyplot as plt
from vfm.data import load_split
a = load_split(CACHE, SPLITS[2], 2000)
fm = np.load(RUN_DIR / f'pred_fm_{SPLITS[2]}_seed0.npy'); yf = np.load(RUN_DIR / f'pred_yflow_{SPLITS[2]}_seed0.npy')
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for j, i in enumerate(np.random.default_rng(0).choice(len(a['fut']), 4, replace=False)):
    for row, (P, name) in enumerate(((fm, 'FM'), (yf, 'Y-Flow'))):
        ax = axes[row, j]
        for l, m in zip(a['lane'][i], a['lane_mask'][i]):
            if m.any(): ax.plot(l[m, 0], l[m, 1], color='0.8', lw=1)
        for k in range(P.shape[1]): ax.plot(P[i, k, :, 0], P[i, k, :, 1], color='tab:blue', alpha=0.5)
        ax.plot(a['hist'][i][a['hist_mask'][i], 0], a['hist'][i][a['hist_mask'][i], 1], 'k.-')
        ax.plot(a['fut'][i][a['fut_mask'][i], 0], a['fut'][i][a['fut_mask'][i], 1], 'r.-')
        ax.set_aspect('equal'); ax.set_title(f'{name} scene {i}')
plt.tight_layout(); plt.savefig(RUN_DIR / 'samples.png', dpi=100); print('saved', RUN_DIR / 'samples.png')